# API tìm kiếm thời trang

Notebook này gom toàn bộ backend tìm kiếm thời trang vào một tệp duy nhất.

- `POST /api/v1/search/image`: tải ảnh lên -> mã hóa bằng FashionCLIP -> tìm top-k bằng FAISS.
- `POST /api/v1/search/text`: nhận mô tả văn bản -> mã hóa bằng FashionCLIP -> tìm top-k bằng FAISS.
- Có thêm bí danh `/search/image` và `/search/text`.
- Swagger: `http://127.0.0.1:8080/docs`

In [13]:
# Chạy cell này một lần nếu môi trường notebook chưa có thư viện.
# %pip install -r requirements.txt

from pathlib import Path
import os

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'embeddings').exists():
    project_candidates = [path for path in PROJECT_DIR.iterdir() if path.is_dir()]
    project_candidates = [path for path in project_candidates if (path / 'embeddings').exists()]
    if len(project_candidates) == 1:
        PROJECT_DIR = project_candidates[0]

if not (PROJECT_DIR / 'embeddings').exists():
    raise FileNotFoundError(
        'Không tìm thấy thư mục embeddings. Hãy mở notebook từ thư mục chứa dữ liệu backend.'
    )

os.chdir(PROJECT_DIR)
print(f'Thư mục dự án: {PROJECT_DIR}')

Thư mục dự án: c:\Users\Nguyen Ho Vinh  Hien\Downloads\DUAN\fashion_search_backend\fashion_search_backend\week36


In [14]:
import io
import threading

import faiss
import numpy as np
import torch
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import JSONResponse
from PIL import Image
from pydantic import BaseModel, Field
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = 'fashionclip'
EMBED_DIR = PROJECT_DIR / 'embeddings'
BUILT_INDEX_PATH = EMBED_DIR / f'{MODEL_NAME}_image_faiss.index'
EXISTING_INDEX_PATH = PROJECT_DIR / f'{MODEL_NAME}_image_embeddings_flat.index'
INDEX_PATH = BUILT_INDEX_PATH if BUILT_INDEX_PATH.exists() else EXISTING_INDEX_PATH
PRODUCT_IDS_PATH = EMBED_DIR / 'product_ids.npy'
IMAGE_URL_TEMPLATE = '/dataset2/images/{}.jpg'

if not INDEX_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy FAISS index: {INDEX_PATH}')
if not PRODUCT_IDS_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy product IDs: {PRODUCT_IDS_PATH}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Model: {MODEL_NAME}; device: {DEVICE}')
print(f'Index: {INDEX_PATH}')

Model: fashionclip; device: cpu
Index: c:\Users\Nguyen Ho Vinh  Hien\Downloads\DUAN\fashion_search_backend\fashion_search_backend\week36\fashionclip_image_embeddings_flat.index


In [15]:
MODEL = CLIPModel.from_pretrained('patrickjohncyh/fashion-clip').to(DEVICE)
PROCESSOR = CLIPProcessor.from_pretrained('patrickjohncyh/fashion-clip')
MODEL.eval()

INDEX = faiss.read_index(str(INDEX_PATH))
PRODUCT_IDS = np.load(PRODUCT_IDS_PATH, allow_pickle=True)
if INDEX.ntotal != len(PRODUCT_IDS):
    raise ValueError(f'Index có {INDEX.ntotal} vector nhưng product_ids có {len(PRODUCT_IDS)} phần tử')

print(f'Loaded {INDEX.ntotal:,} vectors, dimension={INDEX.d}, product IDs={len(PRODUCT_IDS):,}')

Loaded 44,419 vectors, dimension=512, product IDs=44,419


In [16]:
def embed_image(pil_image: Image.Image) -> np.ndarray:
    inputs = PROCESSOR(images=pil_image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        features = MODEL.get_image_features(**inputs)
    return features.cpu().numpy().astype('float32')


def embed_text(query: str) -> np.ndarray:
    inputs = PROCESSOR(text=[query], return_tensors='pt', padding=True).to(DEVICE)
    with torch.no_grad():
        features = MODEL.get_text_features(**inputs)
    return features.cpu().numpy().astype('float32')


def search_topk(query_vector: np.ndarray, k: int = 10):
    query_vector = query_vector.astype('float32').reshape(1, -1)
    faiss.normalize_L2(query_vector)
    scores, indices = INDEX.search(query_vector, k)
    return [
        {
            'product_id': str(PRODUCT_IDS[index]),
            'image_url': IMAGE_URL_TEMPLATE.format(PRODUCT_IDS[index]),
            'similarity_score': round(float(score), 4),
        }
        for score, index in zip(scores[0], indices[0])
        if index != -1
    ]

print('Embedding và search functions: OK')

Embedding và search functions: OK


In [17]:
app = FastAPI(title='Fashion Search API - Week 36')

class TextSearchRequest(BaseModel):
    query: str = Field(..., min_length=1)
    top_k: int = Field(10, ge=1, le=50)


@app.exception_handler(HTTPException)
async def http_exception_handler(request, exc):
    return JSONResponse(status_code=exc.status_code, content={'status': 'error', 'message': exc.detail})


@app.post('/search/image')
@app.post('/api/v1/search/image')
async def search_by_image(image: UploadFile = File(...), top_k: int = Form(10, ge=1, le=50)):
    if not image.content_type or not image.content_type.startswith('image/'):
        raise HTTPException(status_code=400, detail='File phải là ảnh (jpg/png/jpeg)')
    contents = await image.read()
    if not contents:
        raise HTTPException(status_code=400, detail='File ảnh rỗng')
    try:
        pil_image = Image.open(io.BytesIO(contents)).convert('RGB')
    except Exception as exc:
        raise HTTPException(status_code=400, detail='Không đọc được ảnh, file có thể bị hỏng') from exc
    return {'status': 'success', 'data': search_topk(embed_image(pil_image), k=top_k)}


@app.post('/search/text')
@app.post('/api/v1/search/text')
async def search_by_text(body: TextSearchRequest):
    return {'status': 'success', 'data': search_topk(embed_text(body.query), k=body.top_k)}


@app.get('/')
async def health_check():
    return {'status': 'ok', 'message': 'Fashion Search API đang chạy'}

print('FastAPI routes: OK')

FastAPI routes: OK


In [18]:
# Khởi động máy chủ nền để notebook vẫn sử dụng được. Chạy cell này một lần.
import uvicorn

API_HOST = '127.0.0.1'
API_PORT = 8080

if 'SERVER_THREAD' not in globals() or not SERVER_THREAD.is_alive():
    SERVER_CONFIG = uvicorn.Config(
        app,
        host=API_HOST,
        port=API_PORT,
        log_level='info',
    )
    SERVER = uvicorn.Server(SERVER_CONFIG)
    SERVER_THREAD = threading.Thread(target=SERVER.run, daemon=True)
    SERVER_THREAD.start()
    print(f'API đang chạy tại http://{API_HOST}:{API_PORT}/docs')
else:
    print(f'API đã chạy tại http://{API_HOST}:{API_PORT}/docs')

API đã chạy tại http://127.0.0.1:8080/docs


INFO:     127.0.0.1:55596 - "POST /api/v1/search/text HTTP/1.1" 200 OK


## Hướng dẫn sử dụng và kiểm thử API

### 1. Chạy notebook

Chạy các cell theo thứ tự từ trên xuống:

1. Xác định thư mục chứa dữ liệu backend.
2. Tải thư viện, mô hình FashionCLIP, chỉ mục FAISS và danh sách mã sản phẩm.
3. Tạo ứng dụng FastAPI.
4. Chạy cell khởi động máy chủ.

Lần đầu chạy, FashionCLIP có thể mất vài phút để tải mô hình từ Hugging Face.

### 2. Mở Swagger

Sau khi cell khởi động máy chủ chạy thành công, mở:

```text
http://127.0.0.1:8080/docs
```

Các endpoint theo API contract:

- `POST /api/v1/search/image`
- `POST /api/v1/search/text`

### 3. Kiểm thử tìm kiếm bằng văn bản

```powershell
$body = @{
    query = 'áo thun nam màu đen phong cách Y2K'
    top_k = 5
} | ConvertTo-Json

Invoke-RestMethod `
    -Uri http://127.0.0.1:8080/api/v1/search/text `
    -Method Post `
    -ContentType 'application/json' `
    -Body $body
```

Nội dung yêu cầu:

```json
{
  "query": "áo thun nam màu đen phong cách Y2K",
  "top_k": 5
}
```

### 4. Kiểm thử tìm kiếm bằng hình ảnh

Thay đường dẫn ảnh mẫu trong lệnh sau bằng tệp `.jpg`, `.jpeg` hoặc `.png` thật:

```powershell
curl.exe -X POST "http://127.0.0.1:8080/api/v1/search/image" `
    -F "image=@C:\đường-dẫn\đến\ảnh-mẫu.jpg" `
    -F "top_k=5"
```

### 5. Phản hồi thành công

```json
{
  "status": "success",
  "data": [
    {
      "product_id": "42156",
      "image_url": "/dataset2/images/42156.jpg",
      "similarity_score": 0.94
    }
  ]
}
```

`top_k` nhận giá trị từ `1` đến `50`, mặc định là `10`. Nếu tệp tải lên không phải hình ảnh, bị rỗng hoặc bị hỏng, API trả về mã trạng thái `400` cùng thông báo lỗi.